# 02 — Context Management

This notebook explores how context is managed in coding assistants —
Claude.md files, file mentions, conversation control, and how to provide
the right amount of information for effective results.

## Learning Objectives
1. Understand the context hierarchy (machine → project → local)
2. Build a context manager that merges multiple sources
3. Experiment with context size and its effect on response quality

## Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
assert api_key, "Set ANTHROPIC_API_KEY in your .env file"
print("Ready.")

## Context Hierarchy

Claude Code loads context from three levels:

```
Machine Level  (~/.claude/Claude.md)      → Global preferences
Project Level  (./Claude.md)              → Team-shared context
Local Level    (./.claude/Claude.md)      → Personal overrides
```

These are merged together, with local overriding project, which overrides machine.

In [ ]:
def load_context_hierarchy(project_dir: str) -> dict[str, str]:
    """Load Claude.md files from all three levels."""
    contexts = {}
    project = Path(project_dir)

    # Machine level
    machine_path = Path.home() / ".claude" / "Claude.md"
    if machine_path.exists():
        contexts["machine"] = machine_path.read_text()
    else:
        contexts["machine"] = ""

    # Project level
    project_path = project / "Claude.md"
    if project_path.exists():
        contexts["project"] = project_path.read_text()
    else:
        contexts["project"] = ""

    # Local level
    local_path = project / ".claude" / "Claude.md"
    if local_path.exists():
        contexts["local"] = local_path.read_text()
    else:
        contexts["local"] = ""

    return contexts


def merge_contexts(contexts: dict[str, str]) -> str:
    """Merge context from all levels into a single system prompt."""
    parts = []
    for level in ["machine", "project", "local"]:
        content = contexts.get(level, "").strip()
        if content:
            parts.append(f"--- {level.upper()} CONTEXT ---\n{content}")
    return "\n\n".join(parts)


print("Context functions defined.")

## Creating Example Context Files

In [ ]:
# Create a sample project directory with context files
sample_dir = Path("/tmp/context_demo")
sample_dir.mkdir(exist_ok=True)

# Project-level Claude.md
project_md = sample_dir / "Claude.md"
project_md.write_text("""\
# Project: Demo App

## Stack
- Python 3.12, FastAPI, PostgreSQL

## Key Files
- src/db/schema.sql — database schema
- src/api/routes.py — API endpoints

## Conventions
- Use type hints on all functions
- Run `pytest` before committing
""")

# Local-level Claude.md
local_dir = sample_dir / ".claude"
local_dir.mkdir(exist_ok=True)
(local_dir / "Claude.md").write_text("""\
## Personal Notes
- I prefer verbose logging
- My dev database is on port 5433
""")

print("Sample context files created.")

In [ ]:
# Load and merge contexts
contexts = load_context_hierarchy(str(sample_dir))
merged = merge_contexts(contexts)

print("=== Merged Context ===")
print(merged)

## Using Context with Claude

The merged context becomes part of the system prompt sent to Claude.

In [ ]:
import anthropic

client = anthropic.Anthropic()


def ask_with_context(question: str, context: str) -> str:
    """Ask Claude a question with project context."""
    system_prompt = (
        "You are a coding assistant. Use the following project context "
        "to inform your responses:\n\n" + context
    )

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": question}],
    )

    return response.content[0].text


# Ask with context
answer = ask_with_context(
    "What testing framework should I use for this project?",
    merged,
)
print(answer)

## File Mention Simulation

The `@file` feature in Claude Code injects file contents into the context.
Let's simulate that behavior.

In [ ]:
import re


def resolve_file_mentions(message: str, base_dir: str = ".") -> str:
    """Replace @file references with file contents."""
    pattern = r"@(\S+)"
    matches = re.findall(pattern, message)

    for file_ref in matches:
        path = Path(base_dir) / file_ref
        if path.exists():
            content = path.read_text()
            injection = f"\n\n--- Contents of {file_ref} ---\n{content}\n---\n"
            message = message.replace(f"@{file_ref}", injection)
        else:
            message = message.replace(f"@{file_ref}", f"[File not found: {file_ref}]")

    return message


# Test with our sample project
msg = "Review @Claude.md and suggest improvements"
resolved = resolve_file_mentions(msg, str(sample_dir))
print(resolved)

## Conversation Compaction Simulation

The `/compact` command summarizes conversation history while preserving key context.
Let's simulate the concept.

In [ ]:
def compact_conversation(messages: list[dict], client: anthropic.Anthropic) -> list[dict]:
    """Summarize a long conversation into a compact context message."""
    conversation_text = "\n".join(
        f"{m['role']}: {m['content']}" for m in messages
        if isinstance(m["content"], str)
    )

    summary_response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": (
                "Summarize this conversation into a concise context paragraph. "
                "Preserve all key decisions, file names, and technical details:\n\n"
                + conversation_text
            ),
        }],
    )

    summary = summary_response.content[0].text
    return [{"role": "user", "content": f"[Compacted conversation context]\n{summary}"}]


# Example usage with a mock conversation
long_conversation = [
    {"role": "user", "content": "I need to add pagination to the /users endpoint"},
    {"role": "assistant", "content": "I'll read the routes file to understand the current implementation."},
    {"role": "user", "content": "The file is at src/api/routes.py"},
    {"role": "assistant", "content": "I see the endpoint returns all users. I'll add limit/offset parameters."},
    {"role": "user", "content": "Use cursor-based pagination instead of offset"},
    {"role": "assistant", "content": "Good call. I'll use the user ID as cursor."},
]

compacted = compact_conversation(long_conversation, client)
print("Compacted conversation:")
print(compacted[0]["content"])

## Exercise

1. Extend `load_context_hierarchy` to handle missing directories gracefully
2. Create a context manager that tracks token usage across conversation turns
3. Implement a `/clear` simulation that resets conversation state
4. Test how response quality changes with different amounts of context